In [1]:
import ollama

response = ollama.embed(model= "nomic-embed-text", input ="Hello, RAG World")
print(response)

model='nomic-embed-text' created_at=None done=None done_reason=None total_duration=1024552800 load_duration=969866400 prompt_eval_count=6 prompt_eval_duration=None eval_count=None eval_duration=None embeddings=[[0.006452591, 0.013302502, -0.1710114, -0.08304114, -0.019150333, -0.026647346, 0.015756475, 0.013115243, -0.017890312, -0.08344694, 0.014751798, 0.03229744, 0.0380357, 0.052730333, 0.004639316, -0.06451598, -0.012255087, -0.032889232, -0.072743244, -0.0083483085, -0.02946187, -0.073782496, 0.066426404, 0.016183862, 0.109911375, -0.021440506, -0.059083764, -0.00015886918, -0.0075052385, 0.004584642, -0.012855457, -0.040013332, -0.022900818, 0.0023421508, 0.08946131, 0.019024422, 0.03667219, 0.014966931, -0.012483264, 0.03449045, 0.0038899274, 0.056254096, 0.026868818, 0.012054149, 0.06280176, 0.017707277, 0.04640597, 0.0038202314, 0.06929892, -0.04700137, 0.007890733, -0.039310426, 0.027098535, 0.012232563, 0.032034848, -0.0014179655, 0.012229312, -0.03472693, -0.028809551, 0.03

In [2]:
from pypdf import PdfReader

pdf_path = "Databricks.pdf"
reader = PdfReader(pdf_path)
print(f"Number of pages: {len(reader.pages)}")

pages_text = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    pages_text.append(text)

print(pages_text[0][:500])

Number of pages: 142
©2022 Databricks Inc. — All rights reserved 
Certification 
Overview
Databricks Certified Associate 
Data Engineer Exam
1


In [3]:
def chunk_text(text, page_num, chunk_size=800, overlap=150):
    chunks=[]
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append({"text": chunk, "page": page_num})
        start += chunk_size - overlap
    return chunks


all_chunks = []
for i, page_text in enumerate(pages_text):
    if page_text.strip():
        page_chunks = chunk_text(page_text, page_num= i+1)
        all_chunks.extend(page_chunks)

print(f"Total Chunks Created: {len(all_chunks)}")
print("\n----Example Chunk----")
print(all_chunks[0])

Total Chunks Created: 149

----Example Chunk----
{'text': '©2022 Databricks Inc. — All rights reserved \nCertification \nOverview\nDatabricks Certified Associate \nData Engineer Exam\n1', 'page': 1}


In [4]:
import ollama

for chunk in all_chunks:
    response = ollama.embed(model = "nomic-embed-text", input = chunk["text"])
    chunk["embedding"] = response["embeddings"][0]

print("Done embedding all chunks")
print(f"Vector length for first chunk: {len(all_chunks[0]['embedding'])}")

Done embedding all chunks
Vector length for first chunk: 768


In [6]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(name="databricks")

ids = [f"chunk_{i}" for i in range(len(all_chunks))]
embeddings = [chunk["embedding"] for chunk in all_chunks]
documents = [chunk["text"] for chunk in all_chunks]
metadatas = [{"page": chunk["page"]} for chunk in all_chunks]

collection.add(
    ids=ids,
    embeddings = embeddings,
    documents = documents,
    metadatas = metadatas
)

print(f"Stored {collection.count()} chunks in ChromaDB.")

Stored 149 chunks in ChromaDB.


In [7]:
def retrieve(question, n_results=3):
    question_embedding = ollama.embed(model="nomic-embed-text", input=question)["embeddings"][0]
    
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=n_results
    )
    return results

test_question = "What is the Answer to Practice Question 6?"
results = retrieve(test_question)

for i, doc in enumerate(results["documents"][0]):
    page = results["metadatas"][0][i]["page"]
    distance = results["distances"][0][i]
    print(f"--- Match {i+1} (page {page}, distance {distance:.4f}) ---")
    print(doc[:200])
    print()

--- Match 1 (page 132, distance 0.1488) ---
Answer to Practice Question 6
Answer B 

--- Match 2 (page 119, distance 0.3792) ---
Answer to Practice Question 1
Answer B 

--- Match 3 (page 130, distance 0.3886) ---
Answer to Practice Question 5
Answer B 



In [8]:
# Fetch chunk_0 directly by ID to see exactly what got embedded
chunk_0 = collection.get(ids=["chunk_0"], include=["documents", "embeddings"])
print(chunk_0["documents"][0])

©2022 Databricks Inc. — All rights reserved 
Certification 
Overview
Databricks Certified Associate 
Data Engineer Exam
1


In [9]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

question_embedding = ollama.embed(model="nomic-embed-text", input="What is the Answer to Practice Question 6?")["embeddings"][0]
chunk_0_embedding = chunk_0["embeddings"][0]

similarity = cosine_similarity(question_embedding, chunk_0_embedding)
print(f"Similarity between question and chunk_0: {similarity:.4f}")

Similarity between question and chunk_0: 0.4625


In [10]:
all_data = collection.get(ids=None, include=["documents", "embeddings", "metadatas"])

scores = []
for doc, emb, meta in zip(all_data["documents"], all_data["embeddings"], all_data["metadatas"]):
    sim = cosine_similarity(question_embedding, emb)
    scores.append((sim, meta["page"], doc[:80]))

scores.sort(reverse=True, key=lambda x: x[0])

for rank, (sim, page, preview) in enumerate(scores[:8]):
    print(f"Rank {rank+1} | sim={sim:.4f} | page={page} | {preview}")

Rank 1 | sim=0.9256 | page=132 | Answer to Practice Question 6
Answer B 
Rank 2 | sim=0.8104 | page=119 | Answer to Practice Question 1
Answer B 
Rank 3 | sim=0.8057 | page=130 | Answer to Practice Question 5
Answer B 
Rank 4 | sim=0.7905 | page=128 | Answer to Practice Question 4
Answer C 
Rank 5 | sim=0.7629 | page=123 | Answer to Practice Question 3
Answer C 
Rank 6 | sim=0.6492 | page=76 | Answer 
Correct Answer A 
Discussion: 
The correct syntax for this query is Answ
Rank 7 | sim=0.6188 | page=110 | Practice Questions 6 - DLT 
A. All datasets will be updated once and the pipelin
Rank 8 | sim=0.6071 | page=51 | Practice Question 5 – Delta Lake
Which of the following describes Delta Lake? 
A


In [11]:
import re

def clean_text(text):
    # Fix broken ordinals like "5\nth" -> "5th"
    text = re.sub(r'(\d)\s*\n\s*(st|nd|rd|th)\b', r'\1\2', text)
    # Collapse multiple newlines/spaces into single spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

pages_text_clean = [clean_text(page.extract_text()) for page in reader.pages]
print(pages_text_clean[0][:300])

©2022 Databricks Inc. — All rights reserved Certification Overview Databricks Certified Associate Data Engineer Exam 1


In [12]:

all_chunks = []
for i, page_text in enumerate(pages_text_clean):
    if page_text.strip():
        page_chunks = chunk_text(page_text, page_num=i + 1)
        all_chunks.extend(page_chunks)

print(f"Total chunks after cleaning: {len(all_chunks)}")


for chunk in all_chunks:
    response = ollama.embed(model="nomic-embed-text", input=chunk["text"])
    chunk["embedding"] = response["embeddings"][0]

print("Re-embedding done.")


client.delete_collection(name="iit_brochure")
collection = client.get_or_create_collection(name="iit_brochure")

ids = [f"chunk_{i}" for i in range(len(all_chunks))]
embeddings = [chunk["embedding"] for chunk in all_chunks]
documents = [chunk["text"] for chunk in all_chunks]
metadatas = [{"page": chunk["page"]} for chunk in all_chunks]

collection.add(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)
print(f"Rebuilt collection with {collection.count()} chunks.")

Total chunks after cleaning: 149
Re-embedding done.
Rebuilt collection with 149 chunks.


In [14]:
test_question = "What is the Answer to Practice Question 6?"
results = retrieve(test_question, n_results=5)

for i, doc in enumerate(results["documents"][0]):
    page = results["metadatas"][0][i]["page"]
    distance = results["distances"][0][i]
    print(f"--- Match {i+1} (page {page}, distance {distance:.4f}) ---")
    print(doc[:200])
    print()

--- Match 1 (page 132, distance 0.1488) ---
Answer to Practice Question 6 Answer B

--- Match 2 (page 119, distance 0.3792) ---
Answer to Practice Question 1 Answer B

--- Match 3 (page 130, distance 0.3886) ---
Answer to Practice Question 5 Answer B

--- Match 4 (page 128, distance 0.4190) ---
Answer to Practice Question 4 Answer C

--- Match 5 (page 123, distance 0.4742) ---
Answer to Practice Question 3 Answer C



In [16]:
def generate_answer(question, n_results=5):
   
    results = retrieve(question, n_results=n_results)
    retrieved_chunks = results["documents"][0]
    
    context = "\n\n---\n\n".join(retrieved_chunks)
    
    prompt = f"""Here is some context from a document:

{context}

---

Based ONLY on the above context, answer this question: {question}

If the answer isn't clearly in the context, say "I don't have enough information to answer that" instead of guessing."""

    response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"], retrieved_chunks

# Test it
answer, chunks_used = generate_answer("What is the Answer to Practice Question 6?")
print("ANSWER:")
print(answer)

ANSWER:
Answer to Practice Question 6 is **Answer B**.


In [17]:
answer, chunks_used = generate_answer("What is the Answer to Practice Question 6?")

print("--- CHUNKS ACTUALLY SENT TO THE LLM ---\n")
for i, chunk in enumerate(chunks_used):
    print(f"[Chunk {i+1}]")
    print(chunk)
    print()

--- CHUNKS ACTUALLY SENT TO THE LLM ---

[Chunk 1]
Answer to Practice Question 6 Answer B

[Chunk 2]
Answer to Practice Question 1 Answer B

[Chunk 3]
Answer to Practice Question 5 Answer B

[Chunk 4]
Answer to Practice Question 4 Answer C

[Chunk 5]
Answer to Practice Question 3 Answer C



In [18]:
answer, chunks_used = generate_answer("What is the Data Governance % number for this course?")
print(answer)

9%
